# CMU 11-768 Assignment 1 · Building and evaluating an agent harness

I’m Russell, a Pitt student who worked through CMU’s agent-harness assignment independently. I built a coding and chess agent workflow I could run in GitHub Codespaces, then used patches, trajectories, and token reports to study how the harness changes model behavior.

Could an agent repair a chess app, submit a patch, and pass an independent grader? I started there, then varied what the same harness kept in memory and what it showed the model about the board. Finally, I turned a missing trajectory observation into a regression test and used the evaluation feedback to repair the harness itself.

[Repository](https://github.com/xqzr2000/cmu-11-768-assignment-1) · [Original CMU assignment](https://github.com/cmu-agents/assignment-1/blob/main/ASSIGNMENT.md)

| Chapter | Design question | Recorded evidence |
| --- | --- | --- |
| 1 · Coding agent | Can the agent produce a patch an independent grader accepts? | Repair resolved in 13 steps, with 1/1 target test and 9/9 regression tests passing. |
| 2 · Context compaction | What changes when older steps become working memory? | Both Django runs resolved. Compaction cut total tokens 24.7% and increased uncached input 2.6×. |
| 3 · Chess agent | How does the interface change move validity and deliberation? | Listing legal moves eliminated three rejected calls in the short gpt-5-mini comparison. DeepSeek’s output tokens per step rose about sixfold. |
| 4 · Harness evaluation | Can a scorer and regression tests make failures actionable? | Three short baseline trials produced nine accepted moves. A revised logging patch passed 3/3 targeted tests after the first passed 2/3. |

**For an interview**, read the saved outputs on GitHub or run the notebook in a Colab Python 3 CPU runtime. Use the source tour in §6 to open implementation files or inspect an individual function. Setup downloads source and Python dependencies. The demos use scripted model replies and local fixtures and require no API keys, Docker, or Modal.

## 0 · What I adapted and why

| Starting point | My version | What it let me learn |
| --- | --- | --- |
| Modal-managed task environments | A thin `Environment` adapter and a reusable Docker runtime in Codespaces | Trace image construction, command execution, timeouts, port mapping, and teardown through code I can inspect. |
| Course model endpoints | OpenAI-compatible clients for DeepSeek-V4.1-Flash and OpenAI gpt-5-mini, which filled the GPT-OSS-120b slot in the observation experiment | Run experiments with endpoints I could access and see how provider and model choices affect latency, tool use, and cache telemetry. |
| Shared ReAct, coding, compaction, and chess exercises | The course’s agent abstractions and task structure, wired to Docker and pinned testbeds | Follow the same loop across two domains and independently replay the outputs. |
| Three course exercises | Repeated chess trial plans, offline scoring, and an OpenCode repair and comparison workflow | Define evidence before running agents, retain failure outcomes, and test generated patches independently. |

[Learning log](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/Learning-Progress.md)

### Set up the source checkout

The notebook pins the Assignment 1 source revision and installs both workspace packages. The demos use the repository’s implementation with a scripted Chat Completions client, a fake terminal, and an in-memory chess API.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess
import sys
import tomllib

REPO_URL = "https://github.com/xqzr2000/cmu-11-768-assignment-1.git"
SOURCE_COMMIT = "ca87bbef76059287468d82fb37ea01aa9646c31d"
workspace = TemporaryDirectory(prefix="cmu-agent-walkthrough-")
repo = Path(workspace.name) / "assignment-1"

subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], cwd=repo, check=True)
runtime_package = repo / "packages" / "agent-sandbox-runtime"
lock = tomllib.loads((repo / "uv.lock").read_text(encoding="utf-8"))
example_dependencies = {"httpx", "jinja2", "openai", "python-dotenv", "pyyaml", "tqdm"}
pins = [
    f"{package['name']}=={package['version']}"
    for package in lock["package"]
    if package["name"] in example_dependencies
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
     str(repo), str(runtime_package), *pins, "python-chess==1.999"],
    check=True,
)
print(f"Assignment 1 source loaded at {SOURCE_COMMIT[:7]}")

Assignment 1 source loaded at ca87bbe


### Start a live run

1. `make run-code-agent` calls the Python entry point declared in `pyproject.toml`. `cli.run_code_agent` resolves the model, output paths, and task.
2. `Task.load` reads the source commit, Dockerfile, and issue. `verify_source` checks the local checkout and `build_testbed_image` prepares the build context for `DockerImage.build`.
3. `Environment.__init__` starts a `Sandbox` and reads its platform information. `Environment.execute` supplies the working directory and default environment, then delegates to the Docker runtime. The CLI owns cleanup via the environment’s context manager.
4. `Agent.run` asks the model to act, dispatches the tool, and puts its observation in the next prompt. Independent graders and reports consume the saved artifacts after the run.

The demos begin at the next boundary. `scripted_agent` feeds SDK response objects into the checked-out code and `ToyTerminal` returns fixed outputs for known commands, so we can follow each tool-call ID by hand.

[Entry points](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py) · [Environment adapter](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/env.py) · [Docker runtime](https://github.com/xqzr2000/cmu-11-768-assignment-1/tree/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime)

In [2]:
import base64
from copy import deepcopy
import json
import os
import runpy
from types import SimpleNamespace
from unittest.mock import patch

import chess
import httpx
from IPython.display import Markdown, display
from openai.types.chat import ChatCompletion

from assignment.agent import Agent, ChessAgent, CodeAgent, StepLimitError, format_chess_state
from assignment.cli import _record_final_observations
from assignment.task import Task

output_dir = Path(workspace.name) / "demo-artifacts"
output_dir.mkdir(exist_ok=True)


def tool_call(call_id, name, arguments):
    return {
        "id": call_id,
        "type": "function",
        "function": {
            "name": name,
            "arguments": arguments if isinstance(arguments, str) else json.dumps(arguments),
        },
    }


def action(*calls):
    return {"role": "assistant", "content": None, "tool_calls": list(calls)}


def scripted_agent(agent_class, *args, actions, summary=None, **kwargs):
    pending = iter(actions)

    def create(**request):
        if "tools" in request:
            message = next(pending)
        else:
            if summary is None:
                raise AssertionError("Unexpected compaction request")
            message = {"role": "assistant", "content": summary}
        return ChatCompletion.model_validate({
            "id": "chatcmpl-colab-demo",
            "object": "chat.completion",
            "created": 0,
            "model": "scripted-colab",
            "choices": [{
                "index": 0,
                "message": message,
                "finish_reason": "tool_calls" if message.get("tool_calls") else "stop",
            }],
        })

    client = SimpleNamespace(chat=SimpleNamespace(completions=SimpleNamespace(create=create)))
    with patch("assignment.agent.base.OpenAI", return_value=client), patch.dict(
        os.environ, {"OPENAI_API_KEY": "unused-colab-placeholder", "OPENAI_MAX_RETRIES": "0"}
    ):
        return agent_class(*args, model="scripted-colab", **kwargs)


class ToyTerminal:
    cwd = "/testbed"
    machine, release, system, version = "x86_64", "demo", "Linux", "Colab fixture"

    def __init__(self, replies=None):
        self.replies = replies or {}
        self.commands = []
        self.stopped = False

    def execute(self, command, **options):
        self.commands.append((command, options))
        if command in self.replies:
            return {"output": self.replies[command], "returncode": 0}
        return {"output": "Command absent from the demo fixtures", "returncode": 1}

    def stop(self):
        self.stopped = True

## 1 · Trace the coding agent

The chess app tries to ask Black to move after White has already ended the game. I kept the course’s pinned chess app and public grader so I could measure the coding agent independently of what the model claimed to have fixed.

### 1.1 · Build the first request

`CodeAgent.__init__` combines agent instructions, sandbox platform details, and the names and descriptions of available skills. The task text and working directory become the user message. `Agent.build_prompt` assembles the messages and the client receives the tool schemas separately. The skill body arrives later through `invoke_skill`.

The next cell reads the real task and public evaluation manifest. Its scripted agent works on a toy greeting, which keeps the tool exchange short enough to inspect in an interview.

[Task and grader files](https://github.com/xqzr2000/cmu-11-768-assignment-1/tree/ca87bbef76059287468d82fb37ea01aa9646c31d/tasks/chess-terminal-move) · [Coding agent](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py)

In [3]:
task = Task.load(repo / "tasks" / "chess-terminal-move")
skill_path = str(repo / "tasks" / "code-skills")
toy_task = "Update the greeting in hello.py to say Hello, Colab."

toy_patch = (
    "diff --git a/hello.py b/hello.py\n"
    "--- a/hello.py\n+++ b/hello.py\n"
    "@@ -1 +1 @@\n-print('Hello')\n+print('Hello, Colab')\n"
)
terminal = ToyTerminal({
    "python hello.py": "Hello\n",
    "update-greeting-demo": "Updated hello.py in the fixture.",
    "git diff -- hello.py > patch.txt": "",
    "cat patch.txt": toy_patch,
})

code_path = output_dir / "code-trajectory.json"
code_agent = scripted_agent(
    CodeAgent, toy_task, terminal,
    skills_path=skill_path, logs_save_path=str(code_path), step_limit=7,
    actions=[
        {"role": "assistant", "content": "I will inspect the greeting first."},
        action(
            tool_call("bad-json", "execute", "{"),
            tool_call("unknown", "missing_tool", {}),
            tool_call("inspect", "execute", {"command": "python hello.py"}),
        ),
        action(tool_call("skill", "invoke_skill", {"name": "submit-task"})),
        action(tool_call("edit", "execute", {"command": "update-greeting-demo"})),
        action(tool_call("write-patch", "execute", {"command": "git diff -- hello.py > patch.txt"})),
        action(tool_call("review-patch", "execute", {"command": "cat patch.txt"})),
        action(tool_call("done", "send_message", {"summary": "Verified the greeting patch."})),
    ],
)
initial_prompt = code_agent.build_prompt()
print("Assignment task:", task.id, "·", task.problem_statement.splitlines()[0])
print("Initial message roles:", [message["role"] for message in initial_prompt])
print("Tools advertised:", [tool["function"]["name"] for tool in code_agent.tools])
print(code_agent.system_prompt.split("<system_information>", 1)[1].split("</system_information>", 1)[0].strip())
print(code_agent.skills["submit-task"]["metadata"])
assert code_agent.skills["submit-task"]["content"] not in code_agent.system_prompt
public_eval = json.loads((repo / "tasks/chess-terminal-move/public_tests/evaluation.json").read_text())
print("Public grader target:", public_eval["fail_to_pass"][0])
print("Existing behaviors covered:", len(public_eval["pass_to_pass"]))

Assignment task: chess-terminal-move · # Engine crashes after a game-ending human move
Initial message roles: ['system', 'user']
Tools advertised: ['invoke_skill', 'execute', 'send_message']
{
  "machine": "x86_64",
  "release": "demo",
  "system": "Linux",
  "version": "Colab fixture"
}
name: submit-task
description: When the task is complete, take the required steps to submit the solution. DO NOT submit without invoking this skill.
Public grader target: tests/test_chess_server.py::test_checkmating_move_returns_final_state_without_engine_reply
Existing behaviors covered: 9


### 1.2–1.4 · Follow actions, tool observations, and completion

1. `Agent.run` calls `maybe_compact_context` before asking the model for an action. Compaction defaults to off here.
2. `query_language_model` saves a prompt snapshot, asks for an action, and increments `steps_taken`. When the response contains just text, the loop appends a recovery message asking for a tool call.
3. `CodeAgent.execute_tool_calls` validates each call. Bad JSON and unknown tools yield tagged error observations. A valid `execute` goes to `env.execute`.
4. `_link_observations` puts tool replies in call order with matching `tool_call_id` values. Those messages become part of the next request.
5. `invoke_skill` returns the full skill. `send_message` sets `finished` and retrieves the submitted patch. The loop saves the trajectory and the fixture stops. On a live run the CLI releases the environment after patch export.

After the simulated edit, the script creates the patch, reads it back, and submits it in three separate actions, as the skill specifies.

In [4]:
code_agent.run()
code_trace = json.loads(code_path.read_text(encoding="utf-8"))
recovery = code_trace["prompts"][1][-1]
print("\nRecovery message:", recovery["content"])
print("\nObservations in the next request:")
for message in code_trace["prompts"][2][-3:]:
    print(f"{message['tool_call_id']:>9}  {message['content'].replace(chr(10), ' ')}")

skill_reply = next(message for message in code_agent.history if message.get("tool_call_id") == "skill")
print("\nSkill loaded:", skill_reply["content"].splitlines()[1])
print("Illustrative patch:\n" + code_agent.submitted_patch)
print("Finished:", code_agent.finished, "| cleanup:", terminal.stopped, "| steps:", len(code_trace["responses"]))

assert [message["tool_call_id"] for message in code_trace["prompts"][2][-3:]] == [
    "bad-json", "unknown", "inspect",
]
assert terminal.commands[0][0] == "python hello.py"
assert code_agent.submitted_patch == toy_patch and terminal.stopped

[agent] step 1/7: requesting action
[agent] step 1: response contained no parsed tool call; the loop should preserve the response and continue
[agent] step 2/7: requesting action
[agent] step 2: tool call(s): execute, missing_tool, execute
[agent] executing: 'python hello.py'
[agent] command finished: returncode=0
[agent] step 3/7: requesting action
[agent] step 3: tool call(s): invoke_skill
[agent] step 4/7: requesting action
[agent] step 4: tool call(s): execute
[agent] executing: 'update-greeting-demo'
[agent] command finished: returncode=0
[agent] step 5/7: requesting action
[agent] step 5: tool call(s): execute
[agent] executing: 'git diff -- hello.py > patch.txt'
[agent] command finished: returncode=0
[agent] step 6/7: requesting action
[agent] step 6: tool call(s): execute
[agent] executing: 'cat patch.txt'
[agent] command finished: returncode=0
[agent] step 7/7: requesting action
[agent] step 7: tool call(s): send_message

Recovery message: Your last response did not contain a 

### 1.5 · Grade the patch independently

`run_code_agent` exports `patch.txt` from the working sandbox. `make check-part1` calls `scripts/evaluate.py`, which loads the manifest and starts a fresh environment. The evaluation harness applies the candidate and public test patch, runs the test command, parses the statuses, and checks the required groups. FAIL_TO_PASS covers the bug. PASS_TO_PASS covers existing behaviors such as reset, legal moves, and shared API state.

The recorded patch skipped Black’s reply when White’s move already produced a terminal outcome. It resolved in **13 steps**, with **1/1 FAIL_TO_PASS** and **9/9 PASS_TO_PASS**. The same grader later accepted all three CodeAgent trials. That gave me a repeatable way to check the loop’s output against the task.

[Patch evaluator](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py) · [Recorded coding trials](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/Coding-Agent-Comparison.md)

The next cell feeds two illustrative pytest summaries to the real parser and verdict logic. Both summaries include the nine regression passes. Changing the target test to PASSED changes the verdict.

In [5]:
from assignment.eval.harness import EvaluationSpec, Report, TestStatus, parse_pytest_report

spec = EvaluationSpec.load(repo / "tasks/chess-terminal-move/public_tests")
target_test, = spec.fail_to_pass
existing_passes = "\n".join(f"PASSED {name}" for name in spec.pass_to_pass)
verdicts = []
for label, outcome in [("Baseline fixture", "FAILED"), ("Candidate fixture", "PASSED")]:
    statuses = parse_pytest_report(f"{outcome} {target_test}\n{existing_passes}")
    report = Report(
        task_id=spec.task_id,
        patch_applied=outcome == "PASSED",
        results=statuses,
        fail_to_pass={target_test: statuses.get(target_test, TestStatus.MISSING)},
        pass_to_pass={test: statuses.get(test, TestStatus.MISSING) for test in spec.pass_to_pass},
    )
    print(f"{label}: target {statuses[target_test].value}, resolved = {report.resolved}")
    verdicts.append(report.resolved)

assert verdicts == [False, True]

Baseline fixture: target FAILED, resolved = False
Candidate fixture: target PASSED, resolved = True


## 2 · Keep useful context as the transcript grows

**Design.** Run the same Django task and DeepSeek model through the same Docker-backed CodeAgent. Vary the compaction threshold (6,000 vs 0) and record independent patch grades, prompt growth, provider cache hits, and summary overhead. `run_swebench_agent` loads the vendored instance and starts its published image with the `testbed` Conda environment. `scripts/evaluate_swebench.py` grades the exported patch in a fresh copy of that image.

### 2.1–2.2 · Compare full history and working memory

`maybe_compact_context` checks the estimated prompt size and whether enough assistant steps exist to retire an older prefix. The estimate uses serialized message length, with the latest provider prompt count as a calibration point when available. `compact_context` asks for a short working memory from the older steps, preserves the most recent complete assistant/tool group, and inserts the memory after the original system and task messages. The trajectory still holds the original API snapshots and records the summary request separately.

Run the same three actions twice. A threshold of 1 forces a check as soon as there are enough old steps, and the long first observation makes the difference visible. The short summary is scripted so you can inspect exactly what survives.

In [6]:
def run_context_example(threshold):
    context_terminal = ToyTerminal({
        "cat notes.txt": "Target file: hello.py.\n" + "Repeated diagnostic output.\n" * 180,
        "python hello.py": "Hello, Colab\n",
        "cat patch.txt": toy_patch,
    })
    agent = scripted_agent(
        CodeAgent, toy_task, context_terminal,
        step_limit=3, compact_threshold_tokens=threshold,
        actions=[
            action(tool_call("old", "execute", {"command": "cat notes.txt"})),
            action(tool_call("recent", "execute", {"command": "python hello.py"})),
            action(tool_call("done", "send_message", {"summary": "Greeting confirmed."})),
        ],
        summary="Target file: hello.py. The notes identify the file to change.",
    )
    agent.run()
    return agent


full = run_context_example(None)
compact = run_context_example(1)
full_prompt, compact_prompt = full.api_prompts[-1], compact.api_prompts[-1]
event, = compact.compaction_events

print("\nBefore the final action:")
for label, prompt in [("Full", full_prompt), ("Compacted", compact_prompt)]:
    print(f"{label:>9} · {len(prompt)} messages ·", [m["role"] for m in prompt])
print("Estimated message tokens before/after:", event["estimated_tokens_before"], event["estimated_tokens_after"])
print(compact_prompt[2]["content"].split("<working_memory>", 1)[1].split("</working_memory>", 1)[0].strip())

assert compact_prompt[:2] == full_prompt[:2]
assert compact_prompt[-2:] == full_prompt[-2:]
assert compact_prompt[-1]["tool_call_id"] == "recent"
assert all(message.get("tool_call_id") != "old" for message in compact_prompt)
assert any("Repeated diagnostic output" in m["content"] for m in compact.api_prompts[1])
assert event["estimated_tokens_after"] < event["estimated_tokens_before"]

[agent] step 1/3: requesting action
[agent] step 1: tool call(s): execute
[agent] executing: 'cat notes.txt'
[agent] command finished: returncode=0
[agent] step 2/3: requesting action
[agent] step 2: tool call(s): execute
[agent] executing: 'python hello.py'
[agent] command finished: returncode=0
[agent] step 3/3: requesting action
[agent] step 3: tool call(s): send_message
[agent] step 1/3: requesting action
[agent] step 1: tool call(s): execute
[agent] executing: 'cat notes.txt'
[agent] command finished: returncode=0
[agent] step 2/3: requesting action
[agent] step 2: tool call(s): execute
[agent] executing: 'python hello.py'
[agent] command finished: returncode=0
[agent] step 3/3: requesting action
[agent] step 3: tool call(s): send_message

Before the final action:
     Full · 6 messages · ['system', 'user', 'assistant', 'tool', 'assistant', 'tool']
Compacted · 5 messages · ['system', 'user', 'user', 'assistant', 'tool']
Estimated message tokens before/after: 1892 586
Target file: 

The two prompts share the original instructions and the complete latest action and observation. In the compacted prompt, a short memory takes the place of the old action and long tool output. The earlier API snapshot still contains that tool output, so it can be audited later.

### Recorded SWE-bench run · Fewer tokens and fewer cache hits

The [token report](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/reports/token-usage-analysis.md) compares two runs on `django__django-15368` with the same model. Both resolved the task and passed **1/1 FAIL_TO_PASS** and **29/29 PASS_TO_PASS**. The compacted run used a threshold of 6,000 and took 20 action steps plus two summary requests. The baseline took 19 action steps.

The peak prompt below is the largest action prompt. Total tokens also include summary requests. Uncached input is total input minus cached input.

In [7]:
usage = {
    "Compacted": {"peak": 5976, "input": 98644, "cached": 80896, "output": 7159},
    "Full context": {"peak": 11400, "input": 135766, "cached": 128896, "output": 4717},
}
for values in usage.values():
    values["total"] = values["input"] + values["output"]
    values["uncached"] = values["input"] - values["cached"]

compact_usage, full_usage = usage["Compacted"], usage["Full context"]
display(Markdown(
    "| Metric | Compacted | Full context |\n| --- | ---: | ---: |\n"
    + "\n".join(
        f"| {label} | {compact_usage[key]:,} | {full_usage[key]:,} |"
        for label, key in [("Peak action prompt", "peak"), ("Total tokens", "total"),
                           ("Cached input tokens", "cached"), ("Uncached input tokens", "uncached"),
                           ("Output tokens", "output")]
    )
))
total_reduction = 1 - compact_usage["total"] / full_usage["total"]
uncached_ratio = compact_usage["uncached"] / full_usage["uncached"]
cache_break_even = (compact_usage["uncached"] - full_usage["uncached"]) / (
    full_usage["cached"] - compact_usage["cached"]
)
print(f"Total tokens fell {total_reduction:.1%}. Uncached input grew {uncached_ratio:.1f}×.")
print(f"Input-only break-even cached rate: {cache_break_even:.1%} of the uncached rate.")

| Metric | Compacted | Full context |
| --- | ---: | ---: |
| Peak action prompt | 5,976 | 11,400 |
| Total tokens | 105,803 | 140,483 |
| Cached input tokens | 80,896 | 128,896 |
| Uncached input tokens | 17,748 | 6,870 |
| Output tokens | 7,159 | 4,717 |

Total tokens fell 24.7%. Uncached input grew 2.6×.
Input-only break-even cached rate: 22.7% of the uncached rate.


The cache break-even is input-only. The compacted run also generated more output tokens, and charges depend on provider rates. I learned to track context headroom and cost separately. One run per condition gives a case study of prompt and cache behavior. More trials would be needed to estimate reliability or attribute behavioral differences to compaction.

[SWE-bench runner](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py) · [Token analysis code](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_token_usage.py)

## 3 · Reuse the agent loop for chess

**Design.** Reuse ReAct against the chess app from Part 1. For the observation comparison, hold the task, deterministic opponent, 15-step budget, two-hour timeout, and reasoning-history setting fixed. Toggle the legal-move list in the instructions and observations for each model. Record rejected calls, plies, output tokens per step, and elapsed time.

`run_chess_agent` starts `ChessSandbox`, which applies the Part 1 patch and starts the server on a loopback-mapped port. `ChessAgent.__init__` resets the game and builds the first observation. `ChessAgent.execute_tool_calls` dispatches to `chess_tools.py` and permits one board-changing action per response.

### 3.1–3.2 · Observe the board and play a move

A **UCI move** such as `e2e4` gives the start and destination squares. A **FEN** gives the position, side to move, castling and en passant information, and move counters. The real chess agent gets its initial state from `POST /api/reset`. Its `play_move` tool sends UCI to `POST /api/move`, updates `last_state`, and sets `finished` from `game_over`. In the production app, Black replies automatically if the game continues.

The notebook's in-memory API handles the same endpoints using `python-chess`. It gives Black a deterministic reply for reproducibility. The first cell builds a toy server and sends both observation formats the same underlying board. Look for the list of legal actions at the bottom of the second view.

In [8]:
class ChessFixture:
    def __init__(self, initial_fen=chess.STARTING_FEN):
        self.initial_fen = initial_fen
        self.board = chess.Board(initial_fen)
        self.requests = []

    @staticmethod
    def state(board, human_move=None, engine_move=None):
        outcome = board.outcome(claim_draw=True)
        return {
            "fen": board.fen(),
            "squares": {chess.square_name(square): piece.symbol()
                        for square, piece in board.piece_map().items()},
            "turn": chess.COLOR_NAMES[board.turn],
            "status": "Game over" if outcome else "In progress",
            "in_check": board.is_check(),
            "game_over": outcome is not None,
            "legal_moves": sorted(move.uci() for move in board.legal_moves),
            "human_move": human_move,
            "engine_move": engine_move,
        }

    def handle(self, request):
        route = (request.method, request.url.path)
        self.requests.append(route)
        payload = json.loads(request.content) if request.content else {}
        try:
            if route == ("POST", "/api/reset"):
                self.board = chess.Board(self.initial_fen)
            elif route == ("POST", "/api/simulate"):
                candidate = chess.Board(payload["fen"])
                if payload.get("move"):
                    candidate.push_uci(payload["move"])
                return httpx.Response(200, json=self.state(candidate))
            elif route == ("POST", "/api/move"):
                human_move = payload["move"]
                self.board.push_uci(human_move)
                engine_move = None
                if not self.board.is_game_over(claim_draw=True):
                    engine_move = min(move.uci() for move in self.board.legal_moves)
                    self.board.push_uci(engine_move)
                return httpx.Response(200, json=self.state(self.board, human_move, engine_move))
            elif route != ("GET", "/api/state"):
                return httpx.Response(404, json={"detail": "Unknown fixture route"})
        except (ValueError, KeyError) as exc:
            return httpx.Response(400, json={"detail": str(exc)})
        return httpx.Response(200, json=self.state(self.board))

    def client(self):
        return httpx.Client(base_url="http://chess-demo.test", transport=httpx.MockTransport(self.handle))


opening = ChessFixture.state(chess.Board())
board_only = format_chess_state(opening, include_legal_moves=False)
board_and_moves = format_chess_state(opening, include_legal_moves=True)
print("BOARD ONLY")
print(board_only)
print("\nADDED WHEN LEGAL MOVES ARE ENABLED")
print(next(line for line in board_and_moves.splitlines() if line.startswith("legal_moves:")))
assert "legal_moves:" not in board_only
assert len(opening["legal_moves"]) == 20

BOARD ONLY
<chess_state>
status: In progress
turn: white
in_check: False
game_over: False
fen: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1
human_move: (none)
engine_move: (none)
board:
    a b c d e f g h
8 | r n b q k b n r | 8
7 | p p p p p p p p | 7
6 | . . . . . . . . | 6
5 | . . . . . . . . | 5
4 | . . . . . . . . | 4
3 | . . . . . . . . | 3
2 | P P P P P P P P | 2
1 | R N B Q K B N R | 1
    a b c d e f g h
recent_history: (none)
</chess_state>

ADDED WHEN LEGAL MOVES ARE ENABLED
legal_moves: a2a3 a2a4 b1a3 b1c3 b2b3 b2b4 c2c3 c2c4 d2d3 d2d4 e2e3 e2e4 f2f3 f2f4 g1f3 g1h3 g2g3 g2g4 h2h3 h2h4


Try an illegal opening move followed by a legal one. A second live move in the same assistant response is rejected before making an HTTP request, since the first live move and Black's reply can change the position. The third scripted action ends the example with a step-budget stop. A separate fixture starts one move before checkmate to show the terminal condition directly.

The trajectory evaluator matches the call ID to its tool message. The CLI appends `final_observations` because there is no following prompt to capture the last replies. The notebook applies that same helper and uses the real offline evaluator.

In [9]:
def run_chess_example(initial_fen, actions, expected_terminal, label):
    fixture = ChessFixture(initial_fen)
    trajectory_path = output_dir / f"{label}-trajectory.json"
    result_path = output_dir / f"{label}-result.json"
    with fixture.client() as client:
        agent = scripted_agent(
            ChessAgent, ToyTerminal(), http_client=client,
            actions=actions, step_limit=len(actions), include_legal_moves=True,
            logs_save_path=str(trajectory_path),
        )
        try:
            agent.run()
        except StepLimitError:
            print("Reached the scripted step budget.")
        _record_final_observations(trajectory_path, agent)

    assert agent.finished is expected_terminal
    result_path.write_text(json.dumps(agent.last_state), encoding="utf-8")
    evaluate = runpy.run_path(str(repo / "scripts" / "eval_chess_trajectory.py"))["evaluate"]
    return agent, fixture.requests, evaluate(trajectory_path, result_path)


opening_agent, opening_requests, opening_score = run_chess_example(
    chess.STARTING_FEN,
    [
        action(tool_call("illegal", "play_move", {"move": "e2e5"})),
        action(tool_call("white-1", "play_move", {"move": "e2e4"}),
               tool_call("parallel-1", "play_move", {"move": "g1f3"})),
        action(tool_call("white-2", "play_move", {"move": "g1f3"})),
    ],
    expected_terminal=False,
    label="opening",
)
mate_agent, mate_requests, mate_score = run_chess_example(
    "7k/5Q2/6K1/8/8/8/8/8 w - - 0 1",
    [action(tool_call("mate", "play_move", {"move": "f7g7"}))],
    expected_terminal=True,
    label="checkmate",
)
print("\nOpening HTTP requests:", opening_requests)
print("Opening move outcomes:", [(entry["move"], entry["outcome"]) for entry in opening_score["moves"]])
print("Checkmate · game_over:", mate_score["game_over"], "| Black's reply:", mate_agent.last_state["engine_move"])
assert (opening_score["accepted_moves"], opening_score["tool_errors"], opening_score["unknown_outcomes"]) == (2, 2, 0)
assert opening_requests.count(("POST", "/api/move")) == 3
assert mate_score["accepted_moves"] == 1 and mate_agent.last_state["engine_move"] is None

[agent] step 1/3: requesting action
[agent] step 1: tool call(s): play_move
[agent] step 2/3: requesting action
[agent] step 2: tool call(s): play_move, play_move
[agent] step 3/3: requesting action
[agent] step 3: tool call(s): play_move
Reached the scripted step budget.
[agent] step 1/1: requesting action
[agent] step 1: tool call(s): play_move

Opening HTTP requests: [('POST', '/api/reset'), ('POST', '/api/move'), ('POST', '/api/move'), ('POST', '/api/move')]
Opening move outcomes: [('e2e5', 'error'), ('e2e4', 'accepted'), ('g1f3', 'error'), ('g1f3', 'accepted')]
Checkmate · game_over: True | Black's reply: None


The opening example has four `play_move` calls, three server requests, and two accepted moves. One move fails chess validation and the extra parallel call fails the harness's ordering rule. In the checkmate example, the same loop stops immediately because the observation contains `game_over: true`.

### 3.3 · Simulate without changing the live board

`simulate_move(fen, move)` sends the supplied FEN to `POST /api/simulate` and returns a position after **one ply**, a move by one side. You can chain those results to explore both sides. A live `play_move` can advance two plies because it includes Black's reply.

In [10]:
sim_fixture = ChessFixture()
with sim_fixture.client() as client:
    sim_agent = scripted_agent(
        ChessAgent, ToyTerminal(), http_client=client, actions=[], programmatic_tools=True,
        auto_stop_environment=False,
    )
    simulated_reply, = sim_agent.execute_tool_calls([
        tool_call("what-if", "simulate_move", {"fen": opening["fen"], "move": "e2e4"}),
    ])
    simulated = json.loads(simulated_reply["content"])
    print("Simulated position:", simulated["fen"])
    print("After simulate · live side to move:", sim_agent.last_state["turn"])
    sim_agent.execute_tool_calls([tool_call("commit", "play_move", {"move": "e2e4"})])
    print("After play     · Black's response:", sim_agent.last_state["engine_move"])
    print("After play     · live side to move:", sim_agent.last_state["turn"])

assert simulated["turn"] == "black"
assert sim_fixture.requests == [("POST", "/api/reset"), ("POST", "/api/simulate"), ("POST", "/api/move")]
assert sim_agent.last_state["fen"] != simulated["fen"]

Simulated position: rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1
After simulate · live side to move: white
After play     · Black's response: a7a5
After play     · live side to move: white


### 3.4–3.5 · Load a search skill and see the sandbox handoff

The `select-move` skill gives the agent a short list of opening moves, then teaches it to search candidate lines with `simulate_move` and commit once with `play_move`. The production `run_python` tool passes the snippet to the sandbox runner as base64, which keeps shell quoting predictable. The sandbox runner captures `stdout`, `stderr`, and an `error` field. After it returns, `ChessAgent` fetches the live board so the next model request sees the board after any committed move.

In this example, the stand-in records what would be sent to the sandbox and returns a scripted runner response. It also advances the fixture board to represent the one committed move. The code reproduces the handoff and refresh using the actual `ChessAgent` dispatcher.

In [11]:
class ToyPythonSandbox:
    def __init__(self, fixture):
        self.fixture = fixture
        self.calls = []

    def execute(self, command, **options):
        self.calls.append((command, options))
        # Represent the one move the canned snippet commits after its search.
        self.fixture.board.push_uci("e2e4")
        self.fixture.board.push_uci("e7e5")
        result = {"stdout": "Selected e2e4\n", "stderr": "", "error": None}
        return {"returncode": 0, "stdout": json.dumps(result)}


skill_fixture = ChessFixture()
sandbox = ToyPythonSandbox(skill_fixture)
with skill_fixture.client() as client:
    skill_agent = scripted_agent(
        ChessAgent, sandbox, http_client=client, actions=[], programmatic_tools=True,
        skills_path=str(repo / "tasks" / "chess-skills"), auto_stop_environment=False,
    )
    registered = [tool["function"]["name"] for tool in skill_agent.tools]
    loaded, = skill_agent.execute_tool_calls([tool_call("load-skill", "invoke_skill", {"name": "select-move"})])
    fen = skill_agent.last_state["fen"]
    snippet = (
        f"candidate = simulate_move({fen!r}, 'e2e4')\n"
        "play_move('e2e4')\n"
        "print('Selected e2e4')"
    )
    runner_reply, = skill_agent.execute_tool_calls([tool_call("run-search", "run_python", {"code": snippet})])

command, options = sandbox.calls[0]
command_prefix, encoded = command.rsplit(" ", 1)
print("Registered tools:", registered)
print("Loaded skill:", loaded["content"].splitlines()[1])
print("Sandbox command:", command_prefix, "<base64 snippet>", f"(timeout: {options['timeout']}s)")
print("Decoded payload:\n" + base64.b64decode(encoded).decode())
print("Runner result:", runner_reply["content"].split("\n\n", 1)[0])
print("Last API call:", skill_fixture.requests[-1])

assert len(set(registered)) == len(registered)
assert base64.b64decode(encoded).decode() == snippet
assert skill_fixture.requests[-1] == ("GET", "/api/state")
assert skill_agent.last_state["fen"] == skill_fixture.board.fen()
assert "Live board after the snippet:" in runner_reply["content"]

Registered tools: ['invoke_skill', 'play_move', 'simulate_move', 'run_python']
Loaded skill: name: select-move
Sandbox command: python /opt/assignment/sandbox_python.py 8000 <base64 snippet> (timeout: 600s)
Decoded payload:
candidate = simulate_move('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1', 'e2e4')
play_move('e2e4')
print('Selected e2e4')
Runner result: {"stdout": "Selected e2e4\n", "stderr": "", "error": null}
Last API call: ('GET', '/api/state')


The decoded payload is a fixed demonstration of the call sequence. The [recorded skill run](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/README.md#10-running-the-chess-agent) used model-generated snippets to search two plies and commit one move per snippet, with the skill's opening shortcut on the first move. The material-only evaluator tied many quiet moves and broke ties by UCI order, which helps explain why the bishop moved repeatedly.

### Recorded chess experiment · What changed when you exposed legal moves?

The [observation report](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/reports/observation-experiment.md) compares one 15-step run per condition for each model. The project used `gpt-5-mini` in the assignment's GPT-OSS-120b slot.

| Model | Board only | Board + legal moves | Output tokens per step, board only → with list |
| --- | ---: | ---: | ---: |
| DeepSeek-V4.1-Flash | 0/15 rejected | 0/15 rejected | 1,277 → 7,817 |
| gpt-5-mini | 3/15 rejected (20%) | 0/15 rejected | 480 → 249 |

For `gpt-5-mini`, the board-only run included one UCI formatting error, one stale king position after castling, and one move that broke a bishop's movement rule. Its run with the list had zero errors. DeepSeek already made zero errors in both runs, and its recorded output tokens per step rose about sixfold with the list. The runs diverged into different positions and none produced a terminal result. These are observations from four short runs.

I learned to measure action validity and deliberation separately when I change a tool interface. The dedicated full-game attempts also stopped early. The notebook’s one-move checkmate illustrates the stop condition in isolation.

[Chess agent](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py) · [Tool calls](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py) · [Server simulation](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_server.py)

## 4 · From trajectories to harness improvements

### 4.1 · Plan, score, and aggregate chess trials

`run_chess_experiment.build_plan` records the revision, configuration, and output paths. `execute_trial` launches a sandbox, records the last board and trajectory, and calls the offline scorer. `eval_chess_trajectory.evaluate` looks up each move’s observation in the next prompt or the final observations. `aggregate_chess_trials.aggregate` counts outcomes and report coverage. `compare_chess_evaluations.compare` presents configuration differences alongside the metrics.

This cell runs the planner in dry-run mode, then reads the separately committed historical baseline. The baseline used the **unpatched** chess app and three gpt-5-mini trials of three steps each. The completed-game count and step budget bound what the baseline tells us about play.

[Experiment runner](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_chess_experiment.py) · [Committed baseline](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/experiments/eval-harness/baseline-003.json)

In [12]:
plan = json.loads(subprocess.check_output(
    [sys.executable, str(repo / "scripts/run_chess_experiment.py"),
     "--experiment-id", "portfolio-demo", "--model", "gpt-5-mini",
     "--trials", "3", "--step-limit", "3", "--unpatched", "--dry-run"],
    cwd=repo, text=True,
))
print("Dry-run configuration:", plan["model"], "|", plan["environment"])
for trial in plan["trials"]:
    print(trial["trial_id"], "→", trial["report"])

baseline = json.loads((repo / "experiments/eval-harness/baseline-003.json").read_text())
totals = baseline["aggregate"]["summary"]
print()
print("Recorded baseline revision:", baseline["source_commit"][:7])
print("Reports:", baseline["aggregate"]["reports_available"], "/", baseline["sample_count"])
print("Accepted moves:", totals["total_accepted_moves"], "/", totals["total_play_move_attempts"])
print("Completed games:", totals["completed_games"])
assert plan["dry_run"] and totals["total_accepted_moves"] == 9

Dry-run configuration: gpt-5-mini | original chess application, no patch
trial-001 → artifacts/eval-harness/portfolio-demo/trial-001-report.json
trial-002 → artifacts/eval-harness/portfolio-demo/trial-002-report.json
trial-003 → artifacts/eval-harness/portfolio-demo/trial-003-report.json

Recorded baseline revision: 24a03bc
Reports: 3 / 3
Accepted moves: 9 / 9
Completed games: 0


### 4.2 · Turn a logging failure into a regression test

Prompt snapshots are recorded *before* each model call. When the loop ends, there may be no later prompt to capture the last tool responses. I used that gap to design an OpenCode repair task and grade both candidates against the same pinned baseline. The first patch dropped the original `role` field. I added that finding to the task and reran it from the same revision. The second patch copied complete tool messages, and the current chess CLI stores them in a `finally` block. The CLI also captures the last known board when the step limit or a server failure interrupts the run. An invalid move can go back to the model for correction. Repeated connection failures trigger a final state check, then the circuit breaker stops the run before another model request.

| Recorded evaluation | First candidate | Revised candidate |
| --- | ---: | ---: |
| FAIL_TO_PASS | 1/2 | 2/2 |
| PASS_TO_PASS | 1/1 | 1/1 |
| Targeted tests | 2/3 | 3/3 |

The revised candidate also passed 15 broader isolated regression tests. This workflow taught me to turn a trace symptom into a specific acceptance criterion. The measured improvement was in logging correctness.

[Experiment notes, patches, and test records](https://github.com/xqzr2000/cmu-11-768-assignment-1/tree/ca87bbef76059287468d82fb37ea01aa9646c31d/tasks/chess-trajectory-logging/experiment-records)

In [13]:
print("Last action observations as preserved by the chess CLI")
for message in json.loads((output_dir / "opening-trajectory.json").read_text())["final_observations"]:
    print("role:", message["role"], "| tool ID:", message["tool_call_id"])
print("The offline scorer accounted for", len(opening_score["moves"]), "move calls.")

Last action observations as preserved by the chess CLI
role: tool | tool ID: white-2
The offline scorer accounted for 4 move calls.


### 4.3 · Compare two coding agents on the same task

The later comparison uses the same Part 1 task and independent grader for CodeAgent and OpenCode. `compare_coding_agents.py` prepares a directory per trial, invokes the chosen runner, grades any patch, and stores the metrics. `run_opencode.py` selects credentials and starts the sandbox workflow. `OpenCodeRunner` installs or locates the CLI, checks the free-mode environment, probes the model, and extracts the candidate patch. `compare.py` normalizes both agent formats and renders the report.

CodeAgent resolved **3/3** trials. OpenCode’s free model refused requests due to quota, so there were **zero usable OpenCode trials** and the harness comparison remains unanswered. The comparison exposed a credential-forwarding path that was closed in free mode and a quota failure the runner now surfaces explicitly. I kept the manual feedback and independent-test workflow after exploring Harness². The separate Codex CLI experiment remained an exploration.

[Coding comparison](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/Coding-Agent-Comparison.md) · [OpenCode workflow](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/OPENCODE_BACKEND.md) · [Codex CLI retrospective](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/Codex-CLI-Exploration.md)

## 5 · Replay the project in Codespaces

Use the repository’s Codespaces workflow to repeat live runs. Set the model and compatible API endpoint in the Codespace. Live agent runs use provider inference and may incur charges. Give comparison runs distinct artifact names.

| Run order | Repository command | Evidence to open |
| --- | --- | --- |
| 1 · Prepare the pinned chess app | `make setup`, then `make doctor` | Source and dependency checks |
| 2 · Generate and independently grade the Part 1 patch | `make run-code-agent`, then `make check-part1` | `artifacts/fix.patch`, `artifacts/part1-trajectory.json`, and the grader’s verdict |
| 3 · Compare compaction | `make run-swebench-agent COMPACT_THRESHOLD=6000`. Repeat with `COMPACT_THRESHOLD=0`, `SWEBENCH_PATCH=artifacts/django__django-15368-baseline.patch`, and `SWEBENCH_TRAJECTORY=artifacts/django__django-15368-baseline-trajectory.json`. | Grade both with `make check-swebench` using the respective patch, then run `make report-tokens`. |
| 4 · Compare board views | Use `make run-obs-deepseek-no-legal` and `make run-obs-deepseek-legal` with `DEEPSEEK_MODEL=deepseek-flash`. For gpt-5-mini use `make run-obs-gpt-oss-no-legal` and `make run-obs-gpt-oss-legal` with `GPT_OSS_MODEL=gpt-5-mini`. Use the matching provider settings, `STEPS=15`, and `AGENT_STRIP_REASONING=1`. | `make report-obs` reads the named trajectories and results. |
| 5 · Follow the programmatic skill | `make run-python-skill-agent STEPS=6` | `artifacts/part3-python-skill-trajectory.json` and the resulting board |
| 6 · Check code independently of inference | `make test` and `make test-docker` | Unit and integration test results |

The repository tracks experiment summaries, grader definitions, and selected repair records. Historical live trajectories and patches were kept under Git-ignored `artifacts/`. The two full-game attempts remained partial. The work was an independent study outside CMU enrollment.

[Makefile](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/Makefile) · [Repository setup and run instructions](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/README.md)

## 6 · Source tour

Use the index as a reference during a live code walkthrough. Each group follows a project workflow from its entry point to its evidence. Every Python implementation and runner file is linked, including helpers and nested functions. The repository also includes tests, task assets, and the upstream chess app pinned by the task manifest.

The inspection cell opens the indicated function directly from the pinned checkout. Try `Agent._link_observations` to trace tool-call IDs, `ChessAgent._track_server_health` to see the circuit breaker, or `evaluate` in the patch grader to follow the independent replay.

[Test suite](https://github.com/xqzr2000/cmu-11-768-assignment-1/tree/ca87bbef76059287468d82fb37ea01aa9646c31d/tests) · [Sandbox tests](https://github.com/xqzr2000/cmu-11-768-assignment-1/tree/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/tests) · [Task inputs](https://github.com/xqzr2000/cmu-11-768-assignment-1/tree/ca87bbef76059287468d82fb37ea01aa9646c31d/tasks)

### Setup, CLI entry points, and Docker

**[.devcontainer/devcontainer.json](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/.devcontainer/devcontainer.json)**  
Codespaces features and the workspace setup hook.

**[.devcontainer/post-create.sh](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/.devcontainer/post-create.sh)**  
Runs the workspace setup after the development container is created.

**[Makefile](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/Makefile)**  
Entry points for bootstrapping, agent runs, independent grading, reports, and verification.

**[pyproject.toml](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/pyproject.toml)**  
Registers the assignment CLI commands and declares the uv workspace.

**[packages/agent-sandbox-runtime/pyproject.toml](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/pyproject.toml)**  
Registers the runtime CLI and packages Docker lifecycle support separately.

**[scripts/bootstrap_sources.sh](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/bootstrap_sources.sh)**  
Fetches the pinned upstream chess app used by the live task.

**[scripts/bootstrap_sandbox.sh](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/bootstrap_sandbox.sh)**  
Checks that the vendored runtime package is present for older setup flows.

**[scripts/bootstrap_chess_harness.sh](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/bootstrap_chess_harness.sh)**  
Prepares and verifies the pinned harness-improvement baseline.

**[src/assignment/doctor.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/doctor.py)**  
Checks prerequisites before a live run.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/doctor.py#L27-L79) · Reports source, Docker, and model configuration readiness.
- [`_configured`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/doctor.py#L20-L24) · Reads a setting and normalizes empty input.

**[packages/agent-sandbox-runtime/src/agent_sandbox_runtime/cli.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/cli.py)**  
Provides standalone runtime checks.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/cli.py#L12-L38) · Selects the Docker doctor or disposable-container smoke test.

**[src/assignment/cli.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py)**  
Connects Make targets to live agent execution and artifact export.

- [`run_code_agent`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L89-L130) · Load the chess task, run CodeAgent, and export its patch and trajectory.
- [`run_swebench_agent`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L133-L181) · Do the same for a vendored SWE-bench instance in its published image.
- [`run_chess_agent`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L247-L314) · Start the patched chess server, run ChessAgent, and preserve its artifacts.
- [`_add_code_agent_arguments`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L54-L72) · Share argument definitions across coding entry points.
- [`_add_compaction_argument`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L32-L51) · Expose threshold, retained-step count, and summary budget flags.
- [`_compact_threshold`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L25-L29) · Validate the threshold and interpret zero as disabled.
- [`_required`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L75-L79) · Reject missing output paths and other required arguments early.
- [`_model`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L82-L86) · Resolve the model flag or fall back to OPENAI_MODEL.
- [`_record_final_observations`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L215-L244) · Complete the saved chess trajectory after the last model action.
- [`_final_observations`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L194-L212) · Copy the tool messages following the final assistant action.
- [`_write_game_result`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/cli.py#L184-L191) · Save the last known board and early-stop reason.

**[src/assignment/task.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/task.py)**  
Loads the local task definition.

- [`Task.load`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/task.py#L34-L86) · Resolve task metadata, text, source, and image paths.
- [`Task.load.sibling`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/task.py#L62-L66) · Resolve task-local paths relative to the manifest.

**[src/assignment/eval/instances.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/instances.py)**  
Resolves the SWE-bench instance and image.

- [`available`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/instances.py#L91-L93) · List the vendored instances by ID.
- [`load`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/instances.py#L95-L104) · Resolve a named instance to its vendored manifest.
- [`Instance.load`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/instances.py#L58-L81) · Read the pinned instance, paths, and evaluation metadata.
- [`Instance.task_id`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/instances.py#L53-L56) · Expose the instance ID in the shared evaluator interface.
- [`published_images`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/instances.py#L83-L89) · Return the known mapping of instance IDs to published image references.

**[src/assignment/utils/image.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/utils/image.py)**  
Turns the pinned task source into an image recipe.

- [`build_testbed_image`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/utils/image.py#L87-L98) · Verify the source and assemble a Dockerfile recipe with dependency pins.
- [`verify_source`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/utils/image.py#L42-L85) · Check the local revision and any relevant worktree changes.
- [`verify_source.complain`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/utils/image.py#L55-L58) · Raise or warn about source mismatches according to strict mode.
- [`_git`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/utils/image.py#L35-L36) · Run a Git command in the source checkout.
- [`is_ignored`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/utils/image.py#L38-L40) · Identify generated files and paths to exclude.

**[packages/agent-sandbox-runtime/src/agent_sandbox_runtime/image.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/image.py)**  
Stages and builds task images.

- [`DockerImage.from_dockerfile`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/image.py#L67-L76) · Create a recipe for a local Dockerfile and source context.
- [`DockerImage.add_local_file`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/image.py#L78-L84) · Return an updated recipe with an additional staged file.
- [`DockerImage.build`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/image.py#L86-L134) · Stage the context, derive an image tag, and build or reuse the image.
- [`_copy_safe`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/image.py#L23-L38) · Copy source into the build context with the runtime’s exclusions.
- [`_fingerprint`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/image.py#L41-L49) · Hash the staged files for the image tag.

**[src/assignment/env.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/env.py)**  
Presents a task-oriented interface to the Docker runtime.

- [`Environment.__init__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/env.py#L18-L48) · Start the sandbox, optionally select Conda, and read platform metadata.
- [`Environment.activate_conda_env`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/env.py#L53-L61) · Add the testbed interpreter directory to the default PATH.
- [`Environment.__enter__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/env.py#L100-L101) · Hand the running environment to the caller.
- [`Environment.execute`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/env.py#L63-L91) · Merge command settings, delegate execution, and report failures.
- [`Environment.is_alive`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/env.py#L50-L51) · Query whether the sandbox still runs.
- [`Environment.tunnel_url`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/env.py#L93-L95) · Resolve the loopback address of a published port.
- [`Environment.__exit__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/env.py#L103-L104) · Stop the environment when the caller leaves its context.
- [`Environment.stop`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/env.py#L97-L98) · Delegate container teardown.

**[packages/agent-sandbox-runtime/src/agent_sandbox_runtime/sandbox.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/sandbox.py)**  
Owns container lifecycle, command execution, and ports.

- [`Sandbox.__init__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/sandbox.py#L31-L84) · Resolve the image, start a disposable container, and bound its lifetime.
- [`Sandbox.__enter__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/sandbox.py#L159-L160) · Return the running sandbox for context-managed use.
- [`Sandbox.execute`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/sandbox.py#L97-L141) · Run a command inside the container and return output and status.
- [`Sandbox.is_alive`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/sandbox.py#L86-L95) · Inspect Docker to tell whether the container is running.
- [`Sandbox.tunnel_url`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/sandbox.py#L143-L152) · Resolve a loopback-published container port.
- [`Sandbox.__exit__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/sandbox.py#L162-L163) · Stop the container as the context unwinds.
- [`Sandbox.stop`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/sandbox.py#L154-L157) · Remove the disposable container.

**[packages/agent-sandbox-runtime/src/agent_sandbox_runtime/process.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/process.py)**  
Wraps the underlying Docker CLI.

- [`docker`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/process.py#L14-L28) · Run a Docker command and normalize subprocess failures.
- [`docker_ready`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/process.py#L31-L36) · Check whether the CLI and daemon are usable.

### Coding tools, shared loop, and context

**[src/assignment/agent/code_agent.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py)**  
Implements the coding tools and completion condition.

- [`CodeAgent.__init__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py#L67-L120) · Assemble coding instructions, task text, tool schemas, and skill catalog.
- [`CodeAgent.execute_tool_calls`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py#L122-L132) · Dispatch each action call and return a linked observation.
- [`CodeAgent._execute_one`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py#L159-L188) · Select the handler or report an unknown tool.
- [`CodeAgent._available_tools`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py#L134-L135) · List callable tool names for validation feedback.
- [`CodeAgent._parse_arguments`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py#L141-L157) · Decode and validate the tool’s argument object.
- [`CodeAgent._run_execute`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py#L190-L246) · Validate and send the terminal action to Environment.
- [`CodeAgent._invoke_skill`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py#L264-L273) · Return the selected skill instructions to the model.
- [`CodeAgent._send_message`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py#L248-L262) · Read the produced patch and mark the coding agent finished.
- [`CodeAgent._error`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/code_agent.py#L137-L139) · Wrap a problem as a recoverable observation.

**[src/assignment/agent/tools.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/tools.py)**  
Declares the JSON tool schemas advertised to each agent.

**[src/assignment/prompts.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/prompts.py)**  
Provides chess instructions for each observation and tool configuration.

**[src/assignment/agent/base.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py)**  
Owns shared prompt construction, action sequencing, compaction, and trajectory snapshots.

- [`Agent.__init__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L112-L197) · Set up the model client, prompt settings, history, and tool registry.
- [`Agent.load_skills`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L199-L231) · Discover skill folders and index their metadata and bodies.
- [`Agent._parse_skill_frontmatter`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L233-L260) · Validate the required YAML name and description.
- [`Agent.run`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L548-L597) · Enforce the step budget and run action, observation, logging, and cleanup.
- [`Agent.maybe_compact_context`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L514-L546) · Gate compaction by budget and history length, and log the event.
- [`Agent.compaction_enabled`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L392-L396) · Tell the loop whether a threshold is configured.
- [`Agent.estimate_active_prompt_tokens`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L375-L390) · Estimate size using message length and recent provider usage.
- [`rough_message_tokens`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L102-L106) · Estimate token count from serialized message length.
- [`Agent.compact_context`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L398-L434) · Summarize older turns and preserve complete recent turns in history.
- [`Agent._split_steps`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L359-L373) · Group assistant turns with their following observations.
- [`Agent._compaction_request`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L437-L459) · Combine objective, prior working memory, and old exchanges.
- [`Agent._render_message`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L468-L489) · Render an old message for the summarizer.
- [`Agent._clip`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L461-L466) · Bound long excerpts by keeping the beginning and end.
- [`Agent._summary_text`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L491-L497) · Read the returned summary text.
- [`Agent._fallback_summary`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L499-L512) · Preserve an action outline if the summary response is empty.
- [`Agent.query_language_model`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L262-L306) · Save the request, call Chat Completions, and record the response.
- [`Agent.build_prompt`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L313-L329) · Assemble system, task, optional working memory, and retained history.
- [`Agent._memory_message`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L331-L337) · Label summarized context for the next request.
- [`Agent.process_response`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L308-L311) · Normalize the SDK’s assistant message into a dictionary.
- [`Agent._assistant_message`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L339-L357) · Select replayable assistant fields and normalize content.
- [`Agent.execute_tool_calls`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L632-L639) · Define the dispatch contract implemented by subclasses.
- [`Agent._link_observations`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L599-L630) · Match one ordered tool reply to each tool-call ID.
- [`format_tool_output`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/base.py#L82-L99) · Format and bound terminal output for model consumption.

### Independent patch grading and token evidence

**[scripts/evaluate.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/evaluate.py)**  
Exposes grading for the local task.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/evaluate.py#L22-L56) · Load task and evaluation spec and pass them to the shared grader.

**[scripts/evaluate_swebench.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/evaluate_swebench.py)**  
Exposes grading for a vendored SWE-bench instance.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/evaluate_swebench.py#L23-L46) · Resolve instance and patch and use the shared grader against the published image.

**[src/assignment/eval/harness.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py)**  
Applies patches and grades named tests in a fresh container.

- [`EvaluationSpec.load`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L52-L81) · Read the public test patch, command, and expected test IDs.
- [`evaluate`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L304-L367) · Resolve an image, open a fresh environment, and grade a candidate.
- [`resolve_image`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L279-L302) · Select the published image or build the local task image.
- [`_grade`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L222-L277) · Apply patches, run tests, and form the report from required test statuses.
- [`_apply_patch`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L203-L220) · Stage and apply a patch in the testbed.
- [`_write_remote_file`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L196-L201) · Transfer text to a known path inside the sandbox.
- [`find_testbed_python`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L132-L146) · Locate the testbed Conda interpreter or fall back to PATH.
- [`parse_report`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L179-L183) · Route logs to the parser for the specified test framework.
- [`parse_pytest_report`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L185-L194) · Read per-test statuses from pytest’s summary.
- [`parse_django_report`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L148-L177) · Read per-test statuses from Django’s verbose output and failure headers.
- [`Report.resolved`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L96-L102) · Check that evaluation has no error and every required test passed.
- [`Report.summary`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/harness.py#L104-L118) · Format test counts and the resolved outcome.

**[scripts/report_token_usage.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_token_usage.py)**  
Compares full and compacted model usage.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_token_usage.py#L209-L224) · Read the two trajectories and write the comparison report.
- [`summarize`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_token_usage.py#L33-L67) · Accumulate action usage, summary usage, and prompt size by step.
- [`_usage`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_token_usage.py#L23-L30) · Normalize provider token and cache fields.
- [`render`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_token_usage.py#L85-L206) · Build the side-by-side usage report.
- [`_pct`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_token_usage.py#L70-L73) · Format relative change.
- [`_series_rows`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_token_usage.py#L76-L82) · Align prompt sizes by step for the report.

### Chess server, observations, and programmatic tools

**[src/assignment/chess_sandbox.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py)**  
Builds the chess testbed wrapper and starts its HTTP server.

- [`ChessSandbox.__init__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py#L59-L109) · Launch the task image, apply an optional patch, and open the server client.
- [`_with_assignment_files`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py#L39-L48) · Include the Part 3 server and tool code in the task image.
- [`ChessSandbox._apply_patch`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py#L111-L127) · Apply the candidate app fix before starting the server.
- [`ChessSandbox._start_server`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py#L129-L156) · Start the service and wait for it to answer.
- [`ChessSandbox.client`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py#L162-L166) · Expose the established HTTP client to the chess agent.
- [`ChessSandbox.state`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py#L168-L172) · Read the current game state.
- [`ChessSandbox.reset`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py#L194-L198) · Start a new game and return the opening state.
- [`ChessSandbox.play`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py#L174-L192) · Submit a move and surface the server’s illegal-move response.
- [`ChessSandbox.stop`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py#L200-L205) · Close HTTP access and tear down the environment.
- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_sandbox.py#L207-L232) · Run the interactive chess sandbox entry point.

**[src/assignment/chess_server.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_server.py)**  
Adds one-ply simulation to the upstream chess application.

- [`create_app`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_server.py#L73-L85) · Construct the live game API and register the simulation endpoint.
- [`create_app.simulate`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_server.py#L78-L83) · Validate the request and translate simulation errors for HTTP.
- [`simulate_position`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/chess_server.py#L22-L71) · Reconstruct a board from FEN and optionally apply one legal ply.

**[src/assignment/agent/chess_agent.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py)**  
Specializes ReAct for live game state and chess tools.

- [`ChessAgent.__init__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L90-L188) · Reset the game and build the first prompt and tool registry.
- [`ChessAgent.format_state`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L190-L193) · Render observations using this agent’s legal-move setting.
- [`format_chess_state`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L47-L84) · Format the board, FEN, state and optional moves for the model.
- [`ChessAgent.maybe_compact_context`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L286-L289) · Check that the server is reachable before a summary request.
- [`ChessAgent.query_language_model`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L291-L293) · Check that the server is reachable before an action request.
- [`ChessAgent._stop_if_environment_lost`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L282-L284) · Stop further model calls after the server is declared unreachable.
- [`ChessAgent.execute_tool_calls`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L195-L251) · Dispatch tools and allow one board-changing call per response.
- [`ChessAgent._play`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L299-L310) · Send the live move and turn its response into an observation.
- [`ChessAgent._run_python`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L312-L323) · Run the sandbox action, then refresh the live board.
- [`ChessAgent._update_state`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L295-L297) · Store the latest position and detect a terminal game.
- [`ChessAgent._track_server_health`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_agent.py#L253-L280) · Track repeated connection failures and confirm loss with a state read.

**[src/assignment/agent/chess_tools.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py)**  
Implements chess actions and HTTP observation encoding.

- [`_game_state`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py#L199-L203) · Read or reset the live state via HTTP.
- [`_parse_object`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py#L33-L55) · Accept and validate an argument dictionary or JSON object.
- [`_play_move`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py#L121-L135) · Post a validated UCI argument to the live move endpoint.
- [`_simulate_move`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py#L96-L118) · Send a FEN and optional move to the simulation endpoint.
- [`_call_api`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py#L58-L71) · Serialize a successful response or return tool-level error text.
- [`_request_state`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py#L74-L93) · Validate the HTTP response and decoded state.
- [`_chess_error`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py#L27-L30) · Tag a chess error as a recoverable observation.
- [`_invoke_skill`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py#L182-L196) · Read the requested chess skill’s instructions from the loaded catalog.
- [`_run_python`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/chess_tools.py#L138-L179) · Pass the encoded snippet to the testbed runner and return its output.

**[src/assignment/sandbox_python.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/sandbox_python.py)**  
Runs programmatic chess tools inside the testbed.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/sandbox_python.py#L43-L63) · Decode the snippet, run it with tool bindings, and capture its output and errors.
- [`build_tools`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/sandbox_python.py#L23-L41) · Expose callable chess tools to the snippet.
- [`build_tools.unwrap`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/sandbox_python.py#L26-L33) · Turn the underlying tool response into a dictionary or exception.
- [`build_tools.simulate_move`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/sandbox_python.py#L35-L36) · Explore a position through the same HTTP interface.
- [`build_tools.play_move`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/sandbox_python.py#L38-L39) · Commit a live move through the same HTTP interface.

**[scripts/report_observation_experiment.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_observation_experiment.py)**  
Summarizes the four recorded observation conditions.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_observation_experiment.py#L104-L177) · Load matched trajectories and results and render the report.
- [`analyze`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_observation_experiment.py#L36-L91) · Count invalid moves, board progress, and run telemetry.
- [`_message`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_observation_experiment.py#L25-L27) · Extract the assistant message from a saved response.
- [`_observations`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_observation_experiment.py#L30-L33) · Collect the tool observations from prompt history.
- [`_classify`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/report_observation_experiment.py#L94-L101) · Group tool replies by acceptance or the kind of failure.

### Trial planning, scoring, and aggregation

**[scripts/run_chess_experiment.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_chess_experiment.py)**  
Plans and optionally runs repeated chess trials.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_chess_experiment.py#L353-L446) · Parse mode and config, preflight output locations, then plan or run.
- [`build_plan`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_chess_experiment.py#L24-L92) · Record provenance, agent config, and paths for independent trials.
- [`git`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_chess_experiment.py#L17-L21) · Inspect the repository checkout when constructing the plan.
- [`execute_experiment`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_chess_experiment.py#L224-L349) · Run the planned trials and write the aggregate summary.
- [`execute_experiment.resolve`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_chess_experiment.py#L261-L263) · Resolve a recorded path relative to the experiment root.
- [`execute_trial`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_chess_experiment.py#L97-L219) · Start one isolated trial and preserve its trajectory, result, and status.

**[scripts/eval_chess_trajectory.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/eval_chess_trajectory.py)**  
Scores observed moves against saved evidence.

- [`evaluate`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/eval_chess_trajectory.py#L9-L101) · Match actions to observations and summarize move outcomes and telemetry.
- [`evaluate.observation_for`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/eval_chess_trajectory.py#L16-L29) · Find the linked observation in the next prompt or final observations.

**[scripts/aggregate_chess_trials.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/aggregate_chess_trials.py)**  
Combines experiment reports and exposes missing data.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/aggregate_chess_trials.py#L192-L197) · Load a plan and write or print the aggregate.
- [`aggregate`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/aggregate_chess_trials.py#L31-L189) · Combine outcome counts and behavioral metrics over available reports.
- [`aggregate.resolve`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/aggregate_chess_trials.py#L45-L47) · Resolve a planned output path relative to the repository root.
- [`read_json`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/aggregate_chess_trials.py#L20-L21) · Read a stored JSON report.
- [`nonnegative_int`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/aggregate_chess_trials.py#L24-L28) · Validate a count before including it in totals.

**[scripts/compare_chess_evaluations.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_chess_evaluations.py)**  
Compares two saved chess experiments.

- [`compare`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_chess_evaluations.py#L66-L140) · Report baseline and candidate metrics alongside configuration differences.
- [`load_experiment`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_chess_evaluations.py#L26-L32) · Read an experiment summary for comparison.
- [`summarize`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_chess_evaluations.py#L35-L63) · Normalize a summary to the metrics the comparison uses.

### OpenCode and coding-agent comparison

**[scripts/compare_coding_agents.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py)**  
Runs and independently grades coding-agent trials.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L211-L248) · Route the CLI command to trial execution, import, or reporting.
- [`run_codeagent`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L115-L139) · Run the assignment coding agent for one comparison trial.
- [`import_codeagent`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L142-L156) · Register an existing Part 1 patch and trajectory as a trial.
- [`run_opencode`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L159-L196) · Start an OpenCode comparison trial and capture its result.
- [`report`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L199-L208) · Render comparison results from stored trials.
- [`_prepare`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L63-L68) · Prepare a per-trial output directory.
- [`_base_record`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L107-L112) · Initialize the provenance fields for a trial.
- [`_git_head`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L53-L56) · Record the current harness revision.
- [`_endpoint_host`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L59-L60) · Extract the model endpoint host for the record.
- [`_stream`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L71-L82) · Run the command, stream its output to disk, and measure elapsed time.
- [`_finish`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L98-L104) · Grade available output and persist the trial record.
- [`_evaluate`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/compare_coding_agents.py#L85-L95) · Invoke the grader on a trial’s patch and read its result.

**[scripts/run_opencode.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_opencode.py)**  
Exposes the separate Docker-backed OpenCode workflow.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_opencode.py#L48-L157) · Check options, set up the task sandbox, and probe or run OpenCode.
- [`select_credentials`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_opencode.py#L36-L45) · Enforce the free-only default and select credentials for explicit paid mode.
- [`_write`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/run_opencode.py#L160-L162) · Persist a runner record as JSON.

**[src/assignment/opencode_runner.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py)**  
Runs OpenCode in a prepared task environment.

- [`OpenCodeRunner.__init__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L179-L214) · Set the model, environment, and credential policy.
- [`is_free_opencode_model`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L39-L41) · Match the free-tier OpenCode model naming policy.
- [`credentials_from_host`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L344-L346) · Read explicitly supported credentials for paid mode.
- [`credential_like_names`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L44-L45) · Identify environment variable names that resemble credentials.
- [`OpenCodeRunner.assert_no_credentials_in_container`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L225-L240) · Check the free-mode sandbox environment before work.
- [`OpenCodeRunner.install`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L216-L223) · Install or locate the OpenCode CLI.
- [`OpenCodeRunner.probe`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L242-L258) · Send a small request to check actual provider availability.
- [`OpenCodeRunner.run`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L293-L341) · Invoke OpenCode and return the resulting patch and run metadata.
- [`OpenCodeRunner._extract_patch`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L265-L291) · Collect permitted source diffs from the task worktree.
- [`OpenCodeRunner._git`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L260-L263) · Run a Git operation in the testbed.
- [`safe_source_path`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L148-L168) · Filter candidate paths to source changes eligible for extraction.
- [`_check`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L171-L175) · Convert a failed sandbox command into a runner error.
- [`classify_run_output`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L48-L61) · Map process output and status to a recorded outcome.
- [`redact_credentials`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L349-L357) · Scrub configured credential values from captured output.
- [`OpenCodeRunError.__init__`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L67-L71) · Preserve error output and return code for classification.
- [`CommandEnvironment.execute`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/opencode_runner.py#L135-L136) · Define the environment interface the runner needs.

**[src/assignment/compare.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py)**  
Reads trial metrics and formats the coding-agent comparison.

- [`trial_dir`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L42-L47) · Choose a per-task, agent, and trial directory.
- [`sanitized_env`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L50-L59) · Prepare the environment for the free OpenCode subprocess.
- [`patch_stats`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L62-L75) · Summarize changed files and added or removed lines.
- [`codeagent_metrics`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L82-L107) · Read steps, token counts, and calls from model responses.
- [`opencode_metrics`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L127-L162) · Read analogous metrics from the CLI event stream.
- [`_events`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L110-L124) · Iterate decoded events from the run output.
- [`_int`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L78-L79) · Coerce telemetry to a usable integer count.
- [`evaluation_summary`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L165-L179) · Normalize the shared grader output.
- [`evaluation_summary.count`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L167-L169) · Count successful cases in one test group.
- [`load_records`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L182-L190) · Load saved trial records for a task.
- [`aggregate`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L208-L233) · Combine metric and outcome counts per agent.
- [`_mean`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L203-L205) · Average available values.
- [`render_report`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L236-L282) · Render the aggregate table and per-trial results.
- [`_fmt`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L193-L200) · Format optional numeric values for display.
- [`_ratio`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L285-L286) · Format a test group’s passed and total counts.
- [`update_doc`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/compare.py#L289-L296) · Rewrite the marked results region in the comparison write-up.

### Packaging and repository wiring

**[scripts/package_submission.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/package_submission.py)**  
Collects the original course submission artifacts when requested.

- [`main`](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/scripts/package_submission.py#L36-L68) · Check expected files and assemble the submission archive.

**[src/assignment/__init__.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/__init__.py)**  
Python package marker.

**[src/assignment/agent/__init__.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/agent/__init__.py)**  
Exports the agent classes and shared helpers.

**[src/assignment/eval/__init__.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/src/assignment/eval/__init__.py)**  
Exports the common grading interface.

**[packages/agent-sandbox-runtime/src/agent_sandbox_runtime/__init__.py](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/packages/agent-sandbox-runtime/src/agent_sandbox_runtime/__init__.py)**  
Exports the standalone image and sandbox API.

**[.github/workflows/tests.yml](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/.github/workflows/tests.yml)**  
Runs unit tests and a separate Docker integration job.

In [14]:
import ast


def show_source(relative_path, symbol=None):
    source_path = (repo / relative_path).resolve()
    if not source_path.is_relative_to(repo.resolve()):
        raise ValueError("Choose a file in the assignment checkout.")
    source = source_path.read_text(encoding="utf-8")
    lines = source.splitlines()
    start, end = 1, len(lines)
    if symbol:
        node = ast.parse(source)
        for part in symbol.split("."):
            node = next(
                (child for child in node.body if getattr(child, "name", None) == part),
                None,
            )
            if node is None:
                raise ValueError(f"{symbol!r} is unavailable in {relative_path}.")
        decorators = getattr(node, "decorator_list", [])
        start = min([node.lineno, *(item.lineno for item in decorators)])
        end = node.end_lineno
    print(f"{relative_path} (lines {start}–{end})")
    print()
    for number in range(start, end + 1):
        print(f"{number:>4}  {lines[number - 1]}")


show_source("src/assignment/agent/base.py", "Agent.build_prompt")

src/assignment/agent/base.py (lines 313–329)

 313      def build_prompt(self) -> list[dict[str, Any]]:
 314          """Assemble the messages for the next request from the agent's state.
 315  
 316          Layout: the standing instructions (system), the task (user), the
 317          compacted working memory if any (user), then the retained interaction
 318          history of assistant actions and their linked tool observations. This
 319          method only reads state, so it is safe to call repeatedly.
 320          """
 321  
 322          messages: list[dict[str, Any]] = [
 323              {"role": "system", "content": self.system_prompt},
 324              {"role": "user", "content": self.task_prompt},
 325          ]
 326          if self.working_memory:
 327              messages.append({"role": "user", "content": self._memory_message()})
 328          messages.extend(deepcopy(self.history))
 329          return messages
